<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-02/AI_Modul_2.8_Praktikum_Fungsi_dalam_Pemrograman.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 2.8: Praktikum Fungsi dalam Pemrograman
### Prinsip Modularitas Rekayasa Perangkat Lunak, Resolusi Ruang Lingkup LEGB, Fungsi Murni, dan Pipeline Komputasi Cerdas Perkebunan

---

> **Diktat Terkait:** [AI_Modul_2.8_Fungsi_dalam_Pemrograman.md](../../docs/part-02/AI_Modul_2.8_Fungsi_dalam_Pemrograman.md)  
> **Outputs:** Pustaka fungsi modular teranotasi tipe ketat (PEP 484), skrip eksperimen hierarki resolusi LEGB, serta pipeline pengujian mutu CPO berbasis komposisi fungsi murni sesuai standar SNI 01-2901-2021 dan PORAM.  
> **Outcomes:** Mahasiswa terampil mendekomposisi persoalan analitika pertanian yang kompleks menjadi fungsi-fungsi atomik, mengelola hierarki memori variabel, merancang fungsi murni bebas efek samping, serta memanfaatkan fungsi tingkat tinggi (*higher-order functions*).  
> **Impacts:** Terciptanya arsitektur sistem cerdas pertanian yang modular, mudah diuji secara otomatis (*unit testable*), bebas galat konkurensi, dan siap diskalakan ke framework big data terdistribusi.
>
> **Tujuan Praktikum:**  
> 1. Mendeklarasikan fungsi dengan anotasi tipe data, parameter nilai baku (*default*), serta argumen variabel panjang (`*args` dan `**kwargs`).  
> 2. Memverifikasi hierarki pencarian variabel LEGB (*Local, Enclosing, Global, Built-in*) dan sifat penutupan leksikal (*closure*).  
> 3. Membandingkan perilaku fungsi murni (*pure function*) terhadap fungsi dengan efek samping (*side effects*).  
> 4. Membangun pipeline analisis mutu laboratorium CPO terpadu berbasis komposisi fungsi.  
> 5. Mengimplementasikan 3 tantangan mandiri: kalkulator NDVI murni, estimator pupuk NPK modular, dan generator pipeline transformasi fitur (*Higher-Order Functions*).

## 1. Persiapan Environment & Pemuatan Library
Jalankan sel berikut untuk memuat pustaka matematika, dataclass, anotasi fungsi, dan visualisasi grafik:

In [ ]:
# Mengimpor modul pustaka standar Python
import math
import sys
import time
import enum
from dataclasses import dataclass
from typing import Callable, List, Tuple, Dict, Any, Optional

# Mengimpor modul numerik dan visualisasi
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Konfigurasi gaya visualisasi grafik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("[OK] Seluruh pustaka berhasil diinisialisasi.")
print(f"[INFO] Versi Python Aktif: {sys.version.split()[0]}")

## 2. Anatomi Tanda Tangan Fungsi, Default Parameter, & Argumen Fleksibel (`*args`, `**kwargs`)
Memeriksa bagaimana Python mengikat argumen posisional, parameter bernilai baku, serta pengemasan argumen dinamis ke dalam tuple dan dictionary.

In [ ]:
# Definisi fungsi dengan Type Hinting dan Argumen Dinamis
def rekam_telemetri_stasiun(
    id_stasiun: str,
    suhu_c: float,
    kelembaban_rh: float = 75.0,  # Default parameter
    *sensor_tambahan: float,       # Variadic positional (*args)
    **metadata_lingkungan: str     # Variadic keyword (**kwargs)
) -> Dict[str, Any]:
    """
    Merekam telemetri stasiun cuaca perkebunan dengan parameter fleksibel.
    """
    return {
        "stasiun": id_stasiun,
        "suhu": suhu_c,
        "rh": kelembaban_rh,
        "sensor_ekstra": sensor_tambahan,
        "metadata": metadata_lingkungan
    }

# 1. Pemanggilan standar dengan argumen posisional & default
p1 = rekam_telemetri_stasiun("AWS-AFDELING-1", 28.5)
print("=== PEMANGGILAN 1: NILAI DEFAULT ===")
print(p1)

# 2. Pemanggilan dengan argumen variabel (*args dan **kwargs)
p2 = rekam_telemetri_stasiun(
    "AWS-AFDELING-2",
    29.2,
    82.0,
    1013.25, 4.5, 120.0,  # Tekanan (hPa), Kecepatan Angin (m/s), Radiasi (W/m2) -> *args
    lokasi="Blok Sawit Inti",
    operator="Laboratorium Agroklimat",
    kondisi="Cerah Berawan"
)
print("\n=== PEMANGGILAN 2: VARIADIC ARGS (*args, **kwargs) ===")
print(f"ID Stasiun      : {p2['stasiun']}")
print(f"Suhu & RH       : {p2['suhu']} C, {p2['rh']}%")
print(f"Sensor Ekstra   : {p2['sensor_ekstra']} (Tipe: {type(p2['sensor_ekstra']).__name__})")
print(f"Metadata Lapangan: {p2['metadata']}")

## 3. Investigasi Resolusi Ruang Lingkup LEGB & Model Penutupan Leksikal (*Closure*)
Membuktikan pencarian variabel bertingkat Local $\rightarrow$ Enclosing $\rightarrow$ Global $\rightarrow$ Built-in, serta implementasi *Factory Function* berbasis *Closure*.

In [ ]:
# Variabel Tingkat Global (G)
FAKTOR_KALIBRASI_GLOBAL = 1.05

def buat_kalibrator_spektro(nama_alat: str, bias_alat: float) -> Callable[[float], float]:
    """
    Factory function yang mengembalikan fungsi kalibrasi dengan mengikat
    variabel pada Enclosing Scope (Closure).
    """
    # Variabel Enclosing (E)
    print(f"[FACTORY] Menginisialisasi kalibrator untuk instrumen: '{nama_alat}' (Bias = {bias_alat})")
    
    def kalibrasi_nilai(absorbansi_mentah: float) -> float:
        # Variabel Lokal (L)
        # Mengakses Enclosing (bias_alat) dan Global (FAKTOR_KALIBRASI_GLOBAL)
        nilai_terkoreksi = (absorbansi_mentah + bias_alat) * FAKTOR_KALIBRASI_GLOBAL
        return round(nilai_terkoreksi, 4)
        
    return kalibrasi_nilai

# Pembuatan dua fungsi kalibrator independen melalui Closure
kalibrator_lab_utama = buat_kalibrator_spektro("Spektrofotometer UV-Vis Lab 1", bias_alat=0.015)
kalibrator_lab_lapangan = buat_kalibrator_spektro("Spektrofotometer Portabel Afdeling", bias_alat=-0.008)

nilai_uji = 0.450
print("\n=== HASIL EKSEKUSI CLOSURE FUNCTION ===")
print(f"Nilai Mentah Sensor : {nilai_uji}")
print(f"Hasil Kalibrasi Lab 1 : {kalibrator_lab_utama(nilai_uji)}")
print(f"Hasil Kalibrasi Lab 2 : {kalibrator_lab_lapangan(nilai_uji)}")

# Inspeksi variabel bebas yang tersimpan di dalam closure cell
print(f"Isi Variabel Closure Lab 1: {kalibrator_lab_utama.__closure__[0].cell_contents}")

## 4. Evaluasi Komparatif: Fungsi Murni (*Pure Function*) versus Efek Samping (*Side Effects*)
Membuktikan secara empiris perbedaan stabilitas antara fungsi yang memutasi koleksi input secara langsung versus fungsi murni yang menghasilkan objek baru.

In [ ]:
# 1. Fungsi dengan Efek Samping (Impure Function / Destructive Mutation)
def normalisasi_impure(data: List[float]) -> List[float]:
    for i in range(len(data)):
        data[i] = round(data[i] / 100.0, 3)  # Memodifikasi list asli secara in-place!
    return data

# 2. Fungsi Murni (Pure Function / Non-Destructive)
def normalisasi_pure(data: List[float]) -> List[float]:
    # Menghasilkan list baru tanpa menyentuh list asli
    return [round(x / 100.0, 3) for x in data]

telemetri_asli_1 = [45.0, 78.5, 92.0]
print("=== 1. PENGUJIAN FUNGSI MURNI (PURE) ===")
print(f"Data Sebelum Pemanggilan 1 : {telemetri_asli_1}")
hasil_p1 = normalisasi_pure(telemetri_asli_1)
print(f"Hasil Pemanggilan 1        : {hasil_p1}")
hasil_p2 = normalisasi_pure(telemetri_asli_1)
print(f"Hasil Pemanggilan 2        : {hasil_p2}")
print(f"Data Asli Tetap Utuh       : {telemetri_asli_1} [AMAN]")

telemetri_asli_2 = [45.0, 78.5, 92.0]
print("\n=== 2. PENGUJIAN FUNGSI DENGAN EFEK SAMPING (IMPURE) ===")
print(f"Data Sebelum Pemanggilan 1 : {telemetri_asli_2}")
hasil_i1 = normalisasi_impure(telemetri_asli_2)
print(f"Hasil Pemanggilan 1        : {hasil_i1}")
hasil_i2 = normalisasi_impure(telemetri_asli_2)  # Data rusak terbagi dua kali!
print(f"Hasil Pemanggilan 2        : {hasil_i2} [TERDISTORSI / RUSAK!]")
print(f"Data Asli Rusak Permanen   : {telemetri_asli_2}")

## 5. Implementasi Kasus Terintegrasi: Pipeline Analisis Laboratorium Mutu CPO
Menerapkan arsitektur dekomposisi fungsional modular untuk menguji Asam Lemak Bebas (FFA), kadar air, dan indeks DOBI sesuai standar SNI 01-2901-2021 dan PORAM.

In [ ]:
class PredikatMutuCPO(enum.Enum):
    SUPER_EXPORT_GRADE = "SUPER CPO EKSPOR (PRIMA)"
    STANDAR_DOMESTIK   = "CPO STANDAR INDUSTRI (SNI)"
    SUB_STANDAR        = "SUB-STANDAR (WAJIB RE-REFINE)"

@dataclass(frozen=True)
class HasilUjiSampelCPO:
    id_sampel: str
    kadar_ffa_persen: float
    kadar_air_persen: float
    indeks_dobi: float
    predikat_mutu: PredikatMutuCPO
    keterangan_teknis: str

# 1. Fungsi Murni Atomik: Kadar Asam Lemak Bebas (FFA)
def hitung_kadar_ffa(
    volume_titran_ml: float,
    normalitas_naoh: float = 0.1,
    berat_sampel_gram: float = 10.0,
    berat_molekul_palmitat: float = 25.6
) -> float:
    if berat_sampel_gram <= 0.0:
        raise ValueError("Berat sampel laboratorium harus bernilai positif.")
    kadar = (volume_titran_ml * normalitas_naoh * berat_molekul_palmitat) / berat_sampel_gram
    return round(kadar, 3)

# 2. Fungsi Murni Atomik: Kadar Air Gravimetri
def hitung_kadar_air(
    cawan_isi_awal: float,
    cawan_isi_kering: float,
    cawan_kosong: float
) -> float:
    berat_sampel_awal = cawan_isi_awal - cawan_kosong
    if berat_sampel_awal <= 0.0:
        raise ValueError("Kalkulasi berat sampel awal menghasilkan nilai tak valid.")
    kehilangan = max(0.0, cawan_isi_awal - cawan_isi_kering)
    persen_air = (kehilangan / berat_sampel_awal) * 100.0
    return round(persen_air, 3)

# 3. Fungsi Murni Atomik: Indeks DOBI
def hitung_indeks_dobi(abs_446nm: float, abs_269nm: float) -> float:
    if abs_269nm <= 0.0 or math.isclose(abs_269nm, 0.0):
        raise ZeroDivisionError("Absorbansi 269 nm mendekati nol, DOBI tak terdefinisi.")
    return round(abs_446nm / abs_269nm, 2)

# 4. Fungsi Agregator Komposisi: Evaluasi Standar Mutu
def evaluasi_kelayakan_ekspor_cpo(
    id_sampel: str, ffa_persen: float, air_persen: float, dobi_val: float
) -> HasilUjiSampelCPO:
    if ffa_persen <= 3.0 and air_persen <= 0.15 and dobi_val >= 2.8:
        return HasilUjiSampelCPO(
            id_sampel, ffa_persen, air_persen, dobi_val,
            PredikatMutuCPO.SUPER_EXPORT_GRADE,
            "Memenuhi standar ekspor prima (PORAM Grade 1)."
        )
    if ffa_persen <= 5.0 and air_persen <= 0.25 and dobi_val >= 2.3:
        return HasilUjiSampelCPO(
            id_sampel, ffa_persen, air_persen, dobi_val,
            PredikatMutuCPO.STANDAR_DOMESTIK,
            "Memenuhi ambang batas SNI 01-2901-2021 domestik."
        )
        
    alasan = []
    if ffa_persen > 5.0: alasan.append(f"FFA tinggi ({ffa_persen}% > 5.0%)")
    if air_persen > 0.25: alasan.append(f"Kadar air tinggi ({air_persen}% > 0.25%)")
    if dobi_val < 2.3: alasan.append(f"DOBI rendah ({dobi_val} < 2.3)")
    return HasilUjiSampelCPO(
        id_sampel, ffa_persen, air_persen, dobi_val,
        PredikatMutuCPO.SUB_STANDAR,
        "; ".join(alasan)
    )

# Eksekusi Batch Pengujian
kumpulan_sampel = [
    {"id": "TANGKI-01", "v_tit": 8.5,  "c_awal": 45.820, "c_kering": 45.808, "c_kos": 35.820, "a446": 1.25, "a269": 0.41},
    {"id": "TANGKI-02", "v_tit": 16.2, "c_awal": 48.150, "c_kering": 48.128, "c_kos": 38.150, "a446": 0.98, "a269": 0.40},
    {"id": "TANGKI-03", "v_tit": 24.5, "c_awal": 46.500, "c_kering": 46.460, "c_kos": 36.500, "a446": 0.72, "a269": 0.45},
]

hasil_laporan = []
print("=== AUDIT PENGUJIAN MUTU CPO PELABUHAN ===")
for s in kumpulan_sampel:
    ffa_val = hitung_kadar_ffa(s["v_tit"])
    air_val = hitung_kadar_air(s["c_awal"], s["c_kering"], s["c_kos"])
    dobi_val = hitung_indeks_dobi(s["a446"], s["a269"])
    laporan = evaluasi_kelayakan_ekspor_cpo(s["id"], ffa_val, air_val, dobi_val)
    hasil_laporan.append(laporan)
    print(f"ID: {laporan.id_sampel} | FFA={laporan.kadar_ffa_persen}% | Air={laporan.kadar_air_persen}% | DOBI={laporan.indeks_dobi}")
    print(f"  -> Status: {laporan.predikat_mutu.value} ({laporan.keterangan_teknis})")

## 6. Solusi Tantangan Mandiri (Scaffolded Challenges)
Berikut disajikan implementasi lengkap untuk ketiga tantangan praktikum berjenjang.

In [ ]:
# === TANTANGAN 1: KALKULATOR INDEKS VEGETASI NDVI MURNI ===
def hitung_ndvi(nir: float, red: float) -> float:
    """
    Menghitung Normalized Difference Vegetation Index (NDVI) dari saluran NIR dan RED.
    Fungsi murni: bebas efek samping, deterministik, dan dilindungi guard divide-by-zero.
    """
    penyebut = nir + red
    if math.isclose(penyebut, 0.0):
        return 0.0  # Proteksi pembagian nol jika sensor membaca kegelapan total
        
    ndvi_raw = (nir - red) / penyebut
    # Penjepit rentang matematis [-1.0, 1.0]
    ndvi_clamped = max(-1.0, min(1.0, ndvi_raw))
    return round(ndvi_clamped, 4)

print("=== TANTANGAN 1: UJI FUNGSI MURNI NDVI ===")
print(f"Vegetasi Rimbun (NIR=0.75, RED=0.15) -> NDVI: {hitung_ndvi(0.75, 0.15)}")
print(f"Tanah Terbuka   (NIR=0.25, RED=0.20) -> NDVI: {hitung_ndvi(0.25, 0.20)}")
print(f"Badan Air       (NIR=0.05, RED=0.12) -> NDVI: {hitung_ndvi(0.05, 0.12)}")
print(f"Sensor Gelap    (NIR=0.00, RED=0.00) -> NDVI: {hitung_ndvi(0.00, 0.00)}")

In [ ]:
# === TANTANGAN 2: PIPELINE MODULAR PENAKSIR KEBUTUHAN DOSIS PUPUK NPK ===
def hitung_defisit_hara(kadar_tanah: float, kebutuhan_standar: float) -> float:
    """Menghitung defisit hara per kilogram per hektar."""
    return max(0.0, kebutuhan_standar - kadar_tanah)

def hitung_kebutuhan_pupuk(defisit_hara: float, efisiensi_serapan: float = 0.6) -> float:
    """Menghitung kebutuhan fisik pupuk dengan mempertimbangkan efisiensi serapan akar."""
    if efisiensi_serapan <= 0.0:
        raise ValueError("Efisiensi serapan harus bernilai positif.")
    return round(defisit_hara / efisiensi_serapan, 2)

def agregator_rekomendasi_pemupukan(id_blok: str, target_tbs_ton_ha: float, data_tanah: Dict[str, float]) -> Dict[str, Any]:
    """
    Menyusun rekomendasi pupuk terpadu tanpa memodifikasi dictionary masukan.
    Standar serapan: N=120kg/ha, P=45kg/ha, K=160kg/ha untuk target 25 ton TBS.
    """
    skala_target = target_tbs_ton_ha / 25.0
    standar_n = 120.0 * skala_target
    standar_p = 45.0 * skala_target
    standar_k = 160.0 * skala_target
    
    def_n = hitung_defisit_hara(data_tanah.get("N", 0.0), standar_n)
    def_p = hitung_defisit_hara(data_tanah.get("P", 0.0), standar_p)
    def_k = hitung_defisit_hara(data_tanah.get("K", 0.0), standar_k)
    
    urea_kg = hitung_kebutuhan_pupuk(def_n, efisiensi_serapan=0.55)
    sp36_kg = hitung_kebutuhan_pupuk(def_p, efisiensi_serapan=0.45)
    mop_kg  = hitung_kebutuhan_pupuk(def_k, efisiensi_serapan=0.65)
    
    return {
        "id_blok": id_blok,
        "target_produksi_ton_ha": target_tbs_ton_ha,
        "rekomendasi_urea_kg_ha": urea_kg,
        "rekomendasi_sp36_kg_ha": sp36_kg,
        "rekomendasi_mop_kg_ha": mop_kg,
    }

print("\n=== TANTANGAN 2: REKOMENDASI PEMUPUKAN MODULAR ===")
analisis_tanah_blok_c = {"N": 45.0, "P": 18.0, "K": 95.0}
rekomendasi = agregator_rekomendasi_pemupukan("BLOK-SAWIT-C12", target_tbs_ton_ha=28.0, data_tanah=analisis_tanah_blok_c)
for k, v in rekomendasi.items():
    print(f"{k:<25}: {v}")

In [ ]:
# === TANTANGAN 3: GENERATOR PIPELINE TRANSFORMASI FITUR (HIGHER-ORDER FUNCTIONS) ===
def buat_pipeline_transformasi(*tahapan_fungsi: Callable[[float], float]) -> Callable[[List[float]], List[float]]:
    """
    Higher-order function yang menyusun rantai komposisi fungsi (function composition)
    f_n(...(f_2(f_1(x)))) terhadap kumpulan data secara deterministik.
    """
    def eksekusi_pipeline(kumpulan_data: List[float]) -> List[float]:
        hasil = []
        for x in kumpulan_data:
            val = x
            for fn in tahapan_fungsi:
                val = fn(val)
            hasil.append(round(val, 4))
        return hasil
    return eksekusi_pipeline

# Definisi fungsi-fungsi transformasi atomik independen
def bersihkan_derau(x: float) -> float:
    return max(0.0, x)  # Pangkas nilai negatif noise sensor

def skala_gain(x: float) -> float:
    return x * 1.25     # Kalibrasi penguatan gain fotodetektor

def offset_suhu(x: float) -> float:
    return x - 0.5      # Koreksi pergeseran baseline sensor

# Perakitan pipeline fungsional
pipeline_sensor = buat_pipeline_transformasi(bersihkan_derau, skala_gain, offset_suhu)

raw_sensor = [-2.1, 10.0, 15.5, 20.0, 32.4]
transformed_sensor = pipeline_sensor(raw_sensor)

print("\n=== TANTANGAN 3: HIGHER-ORDER FUNCTION TRANSFORMATION PIPELINE ===")
print(f"Data Mentah Sensor      : {raw_sensor}")
print(f"Data Hasil Transformasi : {transformed_sensor}")

# Visualisasi Transformasi Sinyal Komparatif
plt.figure(figsize=(9, 4.5))
indeks = list(range(len(raw_sensor)))
plt.plot(indeks, raw_sensor, marker='o', lw=1.8, color='steelblue', label='Sinyal Mentah Sensor')
plt.plot(indeks, transformed_sensor, marker='s', lw=1.8, color='forestgreen', linestyle='--', label='Sinyal Terkalibrasi Fungsional')
plt.xlabel('Indeks Titik Sampel', fontweight='bold')
plt.ylabel('Nilai Respon Sensor', fontweight='bold')
plt.title('Transformasi Fitur Sensor Menggunakan Higher-Order Function Composition', fontweight='bold', fontsize=11)
plt.legend()
plt.tight_layout()
plt.savefig('docs/assets/simulasi_transformasi_fungsional_cpo.png', dpi=150)
print("[INFO] Grafik transformasi berhasil disimpan ke 'docs/assets/simulasi_transformasi_fungsional_cpo.png'.")